<a href="https://colab.research.google.com/github/liyenrondon/IA-2/blob/main/entrenandoredesprofundas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, BatchNormalization, Activation
import ipywidgets as widgets
from IPython.display import display, clear_output

print(f"TensorFlow versión: {tf.__version__}")

# 1. CARGAR Y PREPARAR EL DATASET FASHION-MNIST
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Normalización de píxeles al rango [0, 1]
X_train_norm = X_train / 255.0
X_test_norm = X_test / 255.0

# Nombres de las 10 clases de prendas
class_names = ['Camiseta/Top', 'Pantalón', 'Pullover', 'Vestido', 'Abrigo',
               'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín']

print(f"Datos de entrenamiento: {X_train.shape[0]} imágenes de {X_train.shape[1]}x{X_train.shape[2]} píxeles.")
print(f"Datos de prueba: {X_test.shape[0]} imágenes.")

# 2. DEFINICIÓN DE CONTROLES E INTERFAZ
model = None
history = None

# Widgets de arquitectura
w_hidden = widgets.IntSlider(value=2, min=1, max=4, description='Capas Ocultas:')
w_neurons = widgets.IntSlider(value=128, min=32, max=512, step=32, description='Neuronas/Capa:')
w_activation = widgets.Dropdown(options=['relu', 'leaky_relu', 'elu', 'selu', 'swish'], value='relu', description='Activación:')
w_init = widgets.Dropdown(options=['glorot_uniform', 'he_normal', 'lecun_normal'], value='he_normal', description='Inicialización:')

# Widgets de regularización y estabilización
w_bn = widgets.Checkbox(value=True, description='Batch Normalization (BN)')
w_dropout = widgets.FloatSlider(value=0.2, min=0.0, max=0.5, step=0.05, description='Dropout Rate:')
w_clipnorm = widgets.FloatText(value=1.0, description='Clip Norm:')

# Widgets de optimizador y programación de LR
w_optimizer = widgets.Dropdown(options=['adam', 'adamw', 'sgd_momentum', 'nesterov', 'rmsprop', 'nadam'], value='adamw', description='Optimizador:')
w_lr = widgets.Dropdown(options=[0.0001, 0.001, 0.01, 0.1], value=0.001, description='Learning Rate:')
w_scheduler = widgets.Dropdown(options=['Ninguno', 'Reducir por Desempeño', 'Exponencial'], value='Reducir por Desempeño', description='LR Scheduler:')
w_epochs = widgets.IntSlider(value=8, min=1, max=20, description='Épocas:')

btn_train = widgets.Button(description="🚀 Entrenar Red Neuronal", button_style='primary')
out_train = widgets.Output()

# Widgets de inferencia
w_sample_idx = widgets.IntSlider(value=0, min=0, max=len(X_test)-1, description='Imagen Test:')
w_mc_dropout = widgets.Checkbox(value=False, description='Activar MC Dropout')
w_num_mc_samples = widgets.IntSlider(value=20, min=5, max=50, description='Muestras MC:')
out_pred = widgets.Output()

def get_optimizer(name, lr, clipnorm):
    if name == 'adam':
        return tf.keras.optimizers.Adam(learning_rate=lr, clipnorm=clipnorm)
    elif name == 'adamw':
        return tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4, clipnorm=clipnorm)
    elif name == 'sgd_momentum':
        return tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9, clipnorm=clipnorm)
    elif name == 'nesterov':
        return tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=True, clipnorm=clipnorm)
    elif name == 'rmsprop':
        return tf.keras.optimizers.RMSprop(learning_rate=lr, clipnorm=clipnorm)
    elif name == 'nadam':
        return tf.keras.optimizers.Nadam(learning_rate=lr, clipnorm=clipnorm)

# 3. LÓGICA DE ENTRENAMIENTO
def train_network(b):
    global model, history
    with out_train:
        clear_output()
        print("Construyendo modelo...")

        # Arquitectura del Perceptrón Multicapa (MLP)
        model = Sequential()
        model.add(Flatten(input_shape=(28, 28)))

        for _ in range(w_hidden.value):
            model.add(Dense(w_neurons.value, kernel_initializer=w_init.value))

            if w_bn.value:
                model.add(BatchNormalization())

            model.add(Activation(w_activation.value))

            if w_dropout.value > 0:
                model.add(Dropout(w_dropout.value))

        model.add(Dense(10, activation='softmax', kernel_initializer=w_init.value))

        opt = get_optimizer(w_optimizer.value, w_lr.value, w_clipnorm.value)
        model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        callbacks = []
        if w_scheduler.value == 'Reducir por Desempeño':
            callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1))
        elif w_scheduler.value == 'Exponencial':
            callbacks.append(tf.keras.callbacks.LearningRateScheduler(lambda epoch, lr: lr * 0.9, verbose=1))

        print("Entrenando red neuronal...")
        history = model.fit(
            X_train_norm, y_train,
            epochs=w_epochs.value,
            batch_size=64,
            validation_split=0.2,
            callbacks=callbacks,
            verbose=1
        )

        test_loss, test_acc = model.evaluate(X_test_norm, y_test, verbose=0)
        print(f"\n✅ ¡Entrenamiento completado!")
        print(f"🎯 Precisión en datos de prueba: {test_acc * 100:.2f}%\n")

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(history.history['accuracy'], label='Entrenamiento')
        ax1.plot(history.history['val_accuracy'], label='Validación')
        ax1.set_title('Precisión (Accuracy)')
        ax1.set_xlabel('Época')
        ax1.legend()

        ax2.plot(history.history['loss'], label='Entrenamiento')
        ax2.plot(history.history['val_loss'], label='Validación')
        ax2.set_title('Pérdida (Loss)')
        ax2.set_xlabel('Época')
        ax2.legend()
        plt.show()

        predict_interactive()

btn_train.on_click(train_network)

# 4. LÓGICA DE INFERENCIA Y EVALUACIÓN
def predict_interactive(change=None):
    if model is None:
        with out_pred:
            clear_output()
            print("⚠️ Primero debes entrenar la red presionando el botón '🚀 Entrenar Red Neuronal'.")
        return

    idx = w_sample_idx.value
    img = X_test_norm[idx:idx+1]
    actual_label = class_names[y_test[idx]]

    with out_pred:
        clear_output()

        if w_mc_dropout.value:
            predictions = np.array([model(img, training=True)[0].numpy() for _ in range(w_num_mc_samples.value)])
            pred_probs = predictions.mean(axis=0)
            uncertainty = predictions.std(axis=0)
        else:
            pred_probs = model.predict(img, verbose=0)[0]
            uncertainty = np.zeros_like(pred_probs)

        pred_label = class_names[np.argmax(pred_probs)]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))

        ax1.imshow(X_test[idx], cmap='gray')
        ax1.set_title(f"Real: {actual_label}\nPredicción: {pred_label}")
        ax1.axis('off')

        y_pos = np.arange(len(class_names))
        ax2.barh(y_pos, pred_probs, xerr=uncertainty, align='center', color='skyblue', ecolor='red', capsize=3)
        ax2.set_yticks(y_pos)
        ax2.set_yticklabels(class_names)
        ax2.set_xlim([0, 1])
        ax2.set_xlabel('Probabilidad')
        ax2.set_title('Incertidumbre (Monte Carlo)' if w_mc_dropout.value else 'Distribución de Probabilidad')

        plt.tight_layout()
        plt.show()

w_sample_idx.observe(predict_interactive, names='value')
w_mc_dropout.observe(predict_interactive, names='value')
w_num_mc_samples.observe(predict_interactive, names='value')

# 5. MOSTRAR INTERFAZ
tab1 = widgets.VBox([w_hidden, w_neurons, w_activation, w_init])
tab2 = widgets.VBox([w_bn, w_dropout, w_clipnorm])
tab3 = widgets.VBox([w_optimizer, w_lr, w_scheduler, w_epochs])

tabs = widgets.Tab(children=[tab1, tab2, tab3])
tabs.set_title(0, '1. Arquitectura')
tabs.set_title(1, '2. Estabilización')
tabs.set_title(2, '3. Optimizador/LR')

infer_controls = widgets.VBox([w_sample_idx, w_mc_dropout, w_num_mc_samples])

display(
    widgets.HTML("<h2>⚙️ Panel de Entrenamiento e Inferencia MLP</h2>"),
    tabs,
    btn_train,
    out_train,
    widgets.HTML("<h2>🔍 Inferencia e Incertidumbre</h2>"),
    infer_controls,
    out_pred
)

predict_interactive()

TensorFlow versión: 2.20.0
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Datos de entrenamiento: 60000 imágenes de 28x28 píxeles.
Datos de prueba: 10000 imágenes.


HTML(value='<h2>⚙️ Panel de Entrenamiento e Inferencia MLP</h2>')

Button(button_style='primary', description='🚀 Entrenar Red Neuronal', style=ButtonStyle())

Output()

HTML(value='<h2>🔍 Inferencia e Incertidumbre</h2>')

Output()